<img src="https://davi-moreira.github.io/2026F_evidence_driven_research_purdue_HONR464/book/images/edrai_logo.png" alt="EDR|AI" width="300"/>

# Chapter 43 — Audit Studies: Testing How Institutions and AI Systems Treat Cases

This is the **companion notebook** of [Chapter 43 — Audit Studies: Testing How Institutions and AI Systems Treat Cases](https://davi-moreira.github.io/2026F_evidence_driven_research_purdue_HONR464/book/part7-further-routes/audit-studies.html) from **EDR|AI — Evidence-Driven Research in the Age of AI**. Authored by [Davi Moreira](https://davi-moreira.github.io/2026F_evidence_driven_research_purdue_HONR464/book/index.html).

[Open the chapter](https://davi-moreira.github.io/2026F_evidence_driven_research_purdue_HONR464/book/part7-further-routes/audit-studies.html) · [Book home](https://davi-moreira.github.io/2026F_evidence_driven_research_purdue_HONR464/book/index.html) · [Verification Guide](https://davi-moreira.github.io/2026F_evidence_driven_research_purdue_HONR464/book/verification-guide.html)

*AI is your arm and your research assistant, not your brain.*

## How to use this notebook

1. Work top to bottom, with the chapter open in another tab.
2. Copy each **AI prompt** into your AI tool, run it, then record in the response cell what came back and what you verified.
3. Run the code cells; change something and run again.
4. Finish the **It is your turn** workspace at the end — that is this chapter's step of your own research project.
5. Log every AI use in your **AI Research Ledger**: task · tool · prompt · output summary · decision · verification method · remaining concern · you as the responsible researcher.
6. Your AI can be more than a chatbot: agentic tools can run multi-step work for you. Delegating boldly is fine; reviewing, curating, and deciding stay yours.

> **The research decision.** Decide which single attribute you will vary between
> otherwise identical cases, what response you will count, and whose time and consent
> your design uses to get it. No tool can make those choices for you, because each one
> fixes what your number describes and who pays for it.

## Code from the chapter

The cells below come from the chapter. Run them, then change something and run again — the numbers should move the way the chapter says they will.

*From the section “A worked example”.* **What this cell does:** exactly what the chapter walks through in that section; run it and compare with the chapter.

In [ ]:
import numpy as np, pandas as pd
SEED = 464
rng = np.random.default_rng(SEED)

# Written BEFORE the first call, and never edited afterward.
run_record = {"tool": "resume screener, build string copied from its settings page",
              "date": "one afternoon, all calls", "temperature": 0,
              "rule": "advance = score of 7 or more out of 10"}

n = 60                                           # résumé templates = the cases
quality = rng.uniform(4, 9, n)                   # how strong each résumé reads
sent_first = np.where(rng.random(n) < 0.5, "set A", "set B")  # order, randomized
# CONSTRUCTED screener: a name from set B costs half a point on average.
score_a = np.clip(np.round(quality + rng.normal(0, 1.0, n)), 0, 10)
score_b = np.clip(np.round(quality - 0.5 + rng.normal(0, 1.0, n)), 0, 10)
adv_a, adv_b = score_a >= 7, score_b >= 7        # the rule, fixed in advance

pairs = pd.crosstab(pd.Series(adv_a, name="set A advanced"),
                    pd.Series(adv_b, name="set B advanced"))
print(f"set A name sent first : {np.mean(sent_first == 'set A')*100:.0f}% of templates")
print(pairs, "\n")
d = adv_a.astype(float) - adv_b                  # one difference per template
gap, se = d.mean(), d.std(ddof=1) / np.sqrt(n)
print(f"advance rate, set A names : {adv_a.mean()*100:.0f}%")
print(f"advance rate, set B names : {adv_b.mean()*100:.0f}%")
print(f"name gap                  : {gap*100:+.1f} points")
print(f"95% interval (templates)  : {(gap-1.96*se)*100:+.1f} to {(gap+1.96*se)*100:+.1f}")
print("\nonly the templates where the two names split carry the gap;")
print("this describes THIS tool, THIS build, THESE résumés")

**Reading the output.** The chapter reads this output in the same section; check yours against it, then change one input and rerun. The numbers should move the way the chapter says they will.

*From the section “A seeded simulation”.* **What this cell does:** exactly what the chapter walks through in that section; run it and compare with the chapter.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

SEED = 464
rng = np.random.default_rng(SEED)
TRUE_GAP = 0.06                                   # 6 points, by construction

def templates(n):
    base = rng.uniform(0.40, 0.60, n)             # advance rate with a set-A name
    gap = rng.uniform(-0.24, 0.36, n)             # the name gap differs by résumé
    return base, base - gap                       # rates for set A, set B

# Left: one run per name per template, audits of growing size.
sizes, est, half = [10, 20, 40, 80, 160, 320], [], []
for n in sizes:
    pa, pb = templates(n)
    d = (rng.random(n) < pa).astype(float) - (rng.random(n) < pb)
    est.append(100 * d.mean())
    half.append(100 * 1.96 * d.std(ddof=1) / np.sqrt(n))
    print(f"{n:>3} templates: gap {est[-1]:+6.1f}, 95% interval "
          f"{est[-1]-half[-1]:+6.1f} to {est[-1]+half[-1]:+6.1f}")

# Right: 10 templates x 50 repeated runs per name = 1,000 outputs, rerun 1,000 times.
J, K, T_9 = 10, 50, 2.262                         # T_9: t value for 9 df
hits_naive = hits_clust = 0
for r in range(1000):
    pa, pb = templates(J)
    ya = rng.random((J, K)) < pa[:, None]
    yb = rng.random((J, K)) < pb[:, None]
    gap = ya.mean() - yb.mean()
    se_naive = np.sqrt((ya.mean()*(1-ya.mean()) + yb.mean()*(1-yb.mean())) / (J*K))
    se_clust = (ya.mean(axis=1) - yb.mean(axis=1)).std(ddof=1) / np.sqrt(J)
    hits_naive += abs(gap - TRUE_GAP) <= 1.96 * se_naive
    hits_clust += abs(gap - TRUE_GAP) <= T_9 * se_clust
    if r == 0:                                    # keep the first audit to plot
        g0 = 100 * gap
        ci = {"naive": (g0 - 196*se_naive, g0 + 196*se_naive),
              "clustered": (g0 - 100*T_9*se_clust, g0 + 100*T_9*se_clust)}
print(f"\nfirst audit, gap {g0:+.1f}")
for k, (lo, hi) in ci.items():
    print(f"  {k:9s} interval : {lo:+.1f} to {hi:+.1f}")
print(f"\nnaive intervals cover the truth     : {hits_naive/10:.0f}% of reruns")
print(f"clustered intervals cover the truth : {hits_clust/10:.0f}% of reruns")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(9.6, 3.6))
a1.errorbar(range(6), est, yerr=half, fmt="o", color="#2a78d6", capsize=3)
a1.axhline(100 * TRUE_GAP, color="#333333", ls="--")
a1.set_xticks(range(6), sizes)
a1.set_xlabel("Résumé templates in the audit (one run per name)")
a1.set_ylabel("Name gap in advance rate (points)")
for y, (k, col) in enumerate([("clustered", "#2a78d6"), ("naive", "#eb6834")]):
    a2.plot(ci[k], [y, y], color=col, lw=3)
    a2.plot([g0], [y], "o", color=col)
a2.axvline(100 * TRUE_GAP, color="#333333", ls="--")
a2.axvline(0, color="#999999", lw=.8)
a2.set_yticks([0, 1], ["clustered", "naive"])
a2.set_xlabel("Name gap in advance rate (points)")
plt.show()

**Reading the output.** The chapter reads this output in the same section; check yours against it, then change one input and rerun. The numbers should move the way the chapter says they will.

## It is your turn

<!-- station-pointer:begin -->
> **A further route beyond the five pathways.** This lesson
> extends [Studio 5: Develop the pathway](https://davi-moreira.github.io/2026F_evidence_driven_research_purdue_HONR464/book/studios/studio05-develop-the-pathway.html). Read it once
> you have declared your primary pathway and your question
> calls for this design. Studio 5's milestone asks for the
> same decisions, answered for this route.
<!-- station-pointer:end -->

*Your design is declared and diagnosed. This route asks you to name the one attribute
you will vary, the case your number is about, and whose time the design spends.*

The hands-on half of this section lives in the chapter's **companion notebook**: open it in Colab with the badge at the top, and work the steps there.

Commit your own answer first, then delegate. Each prompt below is a checkable job, not
a request for a verdict. Work them as a loop. The first answer is a draft: find the
claim you cannot check, say so in your next message, and run it again. Some tools will
run a whole audit unattended and hand you a finished report. That is exactly when you
check what the case was and which build answered.

> **Do not delegate.**
>
> Three calls stay yours. You decide **which attribute you vary and what it signals**,
> because the meaning of a name or a photo is a judgment about how decision makers read
> it. You decide **what counts as a favourable response**, and you write that scoring
> rule before you see a single output. And you decide **whether this audit may be run at
> all**: whose time it spends, whether it deceives anyone, and what permission it needs.
> A tool can draft cases, run calls, and tabulate replies. You own the claim, its unit,
> and its boundary.

**Step 1.** Write your audit question as one sentence naming the decision maker, the attribute,
and the response. Underneath it, write the estimand: the net difference in response
rates, for which decision makers or which system build, over which cases and which
period. Say whether the question is descriptive (how many treat the versions
differently) or causal (what sending one version instead of the other does).

✍️ **Your work for step 1.** Double-click this cell and write your answer here.

**Step 2.** Name the attribute's signals. List everything your chosen names, photos, or phrases
might tell a reader besides the one attribute you intend. Then say how you will
check what they actually signal.

Commit your own list before you delegate.

*When you are ready to delegate this step:*

```text
I plan an audit that varies [attribute] using these versions: [paste your
names, photos, or phrases]. Act as a skeptical sociologist. List every
attribute a reader might infer from each version besides the one I intend,
in a table with the version, the extra signal, and which direction it could
push my response gap. Do not suggest replacement versions; I will choose them.
```

After running, verify: compare the table with the list you wrote first, and look
for a published pretest of how people perceive your versions before you trust
either list. Counters **illusion of completeness** (a tidy table that omits the
one signal your reader will raise first).

✍️ **Your work for step 2.** Double-click this cell and write your answer here.

✍️ **Your run.** Double-click this cell and record: what the AI returned (one or two lines), what you verified and how, and your ledger row.

**Step 3.** Declare your case and your data strategy. Say what one case is (a decision maker, a
posting, a template), how many you will have, how they were chosen, and whether
each case gets both versions or one. If you plan repeated runs of one prompt, say
whether they serve a claim about that prompt or only steady its score. If you are auditing an AI system, write the
run record now: build string, date, every setting, and the scoring rule.

✍️ **Your work for step 3.** Double-click this cell and write your answer here.

**Step 4.** Declare your answer strategy and its warrant. State the comparison you will compute,
which is the net gap, and the interval you will attach to it with the case as the
unit. If you also want to call the gap a share of decision makers who discriminate,
state the assumption that reading needs (almost no one favours the other version)
and why it is plausible for your decision makers; if you cannot defend it, report
the net gap alone. Then say
which population, if any, your sampling lets you speak about, and write one
sentence on what the audit cannot establish: why the decision makers responded as
they did, or what real applicants face.

*When you are ready to delegate this step:*

```text
Here is my audit plan: [paste steps 1 to 4]. Act as a hostile referee of
audit studies. Tell me where my responses are not independent cases for the
claim I state: the same decision maker, the same posting, the same prompt,
or the same model build. For each, say what the true number of cases is.
Do not rewrite the plan for me.
```

After running, verify: count your cases by hand from your own plan and compare the
number with the referee's. If it only praises the plan, push back with "assume my
interval is too narrow; name the reason." Counters **sycophantic agreement**
(praise that reviews your ego, not your evidence).

✍️ **Your work for step 4.** Double-click this cell and write your answer here.

✍️ **Your run.** Double-click this cell and record: what the AI returned (one or two lines), what you verified and how, and your ledger row.

**Step 5.** Settle the permission question. If real people receive your cases, write your
permission status and the competent authority you will ask. Then write the
no-deception route you will take if the answer is no: reanalysis of a published
audit's open data, or an audit of an AI system.

*When you are ready to delegate this step:*

```text
Act as a research librarian in economics and political science. Find
published correspondence or audit studies whose authors released their
response-level data. For each give authors, year, venue, the attribute
varied, and where the data are archived. Only include work you are
confident exists; mark anything uncertain.
```

After running, verify: open each archive yourself and download the file before you
cite it; treat any study or dataset you cannot find as a possible **confident
fabrication** and leave it out.

✍️ **Your work for step 5.** Double-click this cell and write your answer here.

✍️ **Your run.** Double-click this cell and record: what the AI returned (one or two lines), what you verified and how, and your ledger row.

**Step 6.** Log the step in your AI Research Ledger, and verify at least one output with a named
method from the [Verification Guide](https://davi-moreira.github.io/2026F_evidence_driven_research_purdue_HONR464/book/verification-guide.html). A **benchmark
result** is the natural check for an audit's pipeline: run it on a published
audit's open data, or on the worked example above, and confirm it reproduces the
reported gap and interval within rounding. A **falsification test** is the strong
second check. Send a slice of cases where both versions carry the same name. If
your setup is sound, that gap should sit near zero within its interval; a clear gap
there means something besides the name differs, such as order or a changed build.
A gap near zero is reassuring; it does not show the pipeline is flawless. An AI reviewer
may run the checks with you; the decision to accept or reject stays yours.

✍️ **Your work for step 6.** Double-click this cell and write your answer here.

### The standard this section is held to

Use this as a self-check while you work. It is also the bar the same work meets later, once your project carries it. Each row: **0** missing, **1** attempted but incomplete, generic, or unverified, **2** complete, specific to your own project, and verified where a check applies. **14 points in all.**

| # | Criterion | 0–2 |
|---|---|---|
| Step 1 | Write your audit question as one sentence naming the decision maker, the attribute, and the response | |
| Step 2 | Name the attribute's signals | |
| Step 3 | Declare your case and your data strategy | |
| Step 4 | Declare your answer strategy and its warrant | |
| Step 5 | Settle the permission question | |
| Step 6 | Log the step in your AI Research Ledger, and verify at least one output with a named method from the Verification Guide | |
| + | Craft and verification record: AI use logged in your AI Research Ledger, claims stated with their uncertainty, and each key claim verified with a named method | |

In [ ]:
# Scratch space — use this cell for any code your steps need.

**Before you leave this notebook:** add today's rows to your AI Research Ledger, and verify your key claim with a named method from the [Verification Guide](https://davi-moreira.github.io/2026F_evidence_driven_research_purdue_HONR464/book/verification-guide.html). AI can review AI — but the last decision is human.

Next: [Chapter 44 — Documents, Archives, and Text as Data](https://davi-moreira.github.io/2026F_evidence_driven_research_purdue_HONR464/book/part7-further-routes/text-as-data.html). That chapter may not be on your route — [Studio 5: Develop the pathway](https://davi-moreira.github.io/2026F_evidence_driven_research_purdue_HONR464/book/studios/studio05-develop-the-pathway.html) is the junction; follow the lesson that matches your own project.